# VisionGuard — Helmet/Plate Training (Colab T4, fast + crash-proof)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

This notebook is optimized for speed and **saves checkpoints to Google Drive after every epoch**,
so if Colab disconnects you keep your progress and can resume. Classes: `No-helmet`, `helmet`,
`motorcycle`, `plate`.

**Tip:** open this notebook *from Google Drive* (File → Save a copy in Drive) so Colab auto-saves it.

In [ ]:
# 1) Mount Google Drive (checkpoints survive disconnects)
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/visionguard_helmet'
os.makedirs(SAVE_DIR, exist_ok=True)
print('checkpoints ->', SAVE_DIR)

In [ ]:
!pip -q install ultralytics roboflow
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to T4!')

In [ ]:
# 2) Download the dataset (paste your Roboflow API key)
from roboflow import Roboflow
rf = Roboflow(api_key='PASTE_YOUR_ROBOFLOW_API_KEY')
project = rf.workspace('helmet-and-number-plate-detection-project').project(
    'helmet-and-number-plate-detection-for-motorbike-safety-iityz')
dataset = project.version(2).download('yolov8')
loc = dataset.location

import yaml, os
cfg = yaml.safe_load(open(os.path.join(loc, 'data.yaml')))
cfg.update({'path': loc, 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images'})
yaml.safe_dump(cfg, open(os.path.join(loc, 'data_fixed.yaml'), 'w'))
print('classes:', cfg['names'])

In [ ]:
# 3) Train — FAST settings + per-epoch save to Drive.
#    Speed levers: yolov8n (light), imgsz 512, batch=-1 (auto-fill GPU), cache, early stop.
#    ~2-4 min/epoch on T4 -> a strong model in well under 2 hours (often early-stops sooner).
import shutil, os
from ultralytics import YOLO

RUN = os.path.join(SAVE_DIR, 'run')          # lives on Drive -> survives disconnects
BEST = os.path.join(RUN, 'helmet', 'weights', 'best.pt')

model = YOLO('yolov8n.pt')                    # 'yolov8s.pt' for a bit more accuracy / slower

def _publish(trainer):                         # copy best.pt to a stable Drive path each epoch
    if os.path.exists(BEST):
        shutil.copy(BEST, os.path.join(SAVE_DIR, 'best.pt'))
model.add_callback('on_fit_epoch_end', _publish)

model.train(
    data=os.path.join(loc, 'data_fixed.yaml'),
    epochs=40, imgsz=512, batch=-1, cache=True, device=0, workers=8,
    patience=12, plots=True, project=RUN, name='helmet', exist_ok=True,
)
print('DONE. Best model at:', os.path.join(SAVE_DIR, 'best.pt'))

## If Colab disconnected — RESUME (don't restart from scratch)
Re-run cells 1–2, then run the cell below instead of cell 3.

In [ ]:
# RESUME from the last checkpoint saved on Drive
import os
from ultralytics import YOLO
LAST = os.path.join(SAVE_DIR, 'run', 'helmet', 'weights', 'last.pt')
assert os.path.exists(LAST), 'no checkpoint yet - run cell 3 first'
YOLO(LAST).train(resume=True)

In [ ]:
# 4) Validate + grab the model
from ultralytics import YOLO
import os
m = YOLO(os.path.join(SAVE_DIR, 'best.pt'))
metrics = m.val(data=os.path.join(loc, 'data_fixed.yaml'))
print('mAP50:', metrics.box.map50, ' mAP50-95:', metrics.box.map)
# best.pt is already in your Drive at SAVE_DIR/best.pt.
# Download it and place at VisionGuard/ml/weights/helmet/best.pt, then restart the backend.
from google.colab import files
files.download(os.path.join(SAVE_DIR, 'best.pt'))